# Geocode Addresses using ArcGIS Location Platform

This notebook demonstrates how to use the `geocoding-kit` module with `arcgis` / `arcgis-mapping` to geocode a list of addresses (from `addresses.csv`) and display the results.

**Prerequisites:**
- Install dependencies: `pip install arcgis arcgis-mapping` or if `uv` is installed, just use `uv run jupyter lab`
- Set `ARCGIS_API_KEY` in your environment or `.env` file.

Run the cells sequentially to see geocoding results.

In [22]:
import os
import sys
from pathlib import Path

# If you have not installed the local `geocoding_kit` package, add the repository root to sys.path.
# This allows running the notebook directly from the `examples/` folder.
repo_root = Path().resolve().parent
sys.path.insert(0, os.path.join(str(repo_root), "src"))

In [23]:
import pandas as pd
from arcgis.gis import GIS

from geocoding_kit import ArcGISConfig, PlatformGeocoder
from geocoding_kit.models import AddressInput

In [27]:
# Initialize the config instance
# Reads the API key and the geocoding endpoint
config = ArcGISConfig.from_env()

# Ensure the API key is set (either in the environment or via a .env file)
api_key = os.getenv("ARCGIS_API_KEY")
if not api_key:
    raise ValueError("Missing ARCGIS_API_KEY environment variable. Set it before running the notebook.")

# Create a GIS instance (optional, used to show how to use arcgis and arcgis-mapping)
portal = GIS(api_key=api_key)
map_view = portal.map("Bonn, Germany")
map_view.basemap.basemap = "osm"
map_view

Map()

## Load Address Data

Load the sample addresses from `addresses.csv`. The file must include at least the columns: `id`, `address`, `postal`, `city`, and `country`.

In [60]:
address_csv = "addresses.csv"

address_df = pd.read_csv(address_csv)
address_df

,id,address,postal,city,country
0,10178,Karl-Liebknecht-Straße 5,10178,Berlin,DE
1,53113,Adenauerallee 206,53113,Bonn,DE
2,20097,Wendenstraße 8 - 12,20097,Hamburg,DE
3,30173,Freundallee 23,30173,Hannover,DE
4,50668,Konrad-Adenauer-Ufer 41-45,50668,Köln,DE
5,85402,Ringstraße 7,85402,Kranzberg,DE
6,4155,Fechnerstraße 8,4155,Leipzig,DE
7,48155,Martin-Luther-King-Weg 24,48155,Münster,DE


## Prepare Inputs for Geocoding

Convert the CSV rows into `AddressInput` instances used by `geocoding-kit`.

In [61]:
addresses = [
    AddressInput(
        id=int(row["id"]),
        address=str(row["address"]),
        postal=str(row.get("postal", "")),
        city=str(row.get("city", "")),
        country=str(row.get("country", "")),
    )
    for _, row in address_df.iterrows()
]

len(addresses), "addresses loaded"

(8, 'addresses loaded')

## Run Geocoding

Use the `PlatformGeocoder` from `geocoding-kit` to geocode the address list.

In [38]:
geocoder = PlatformGeocoder(config)
results = geocoder.geocode(addresses)

results_df = pd.DataFrame([
    {
        "id": result.id,
        "matched_address": result.matched_address,
        "latitude": result.latitude,
        "longitude": result.longitude,
        "score": result.score,
        "match_status": result.match_status,
        "match_type": result.match_type
    }
    for result in results
])

results_df

,id,matched_address,latitude,longitude,score,match_status,match_type
0,10178,"Karl-Liebknecht-Straße 5, 10178, Berlin, Mitte...",52.519724,13.404016,100.00,M,PointAddress
1,53113,"Adenauerallee 206, 53113, Bonn, Gronau, Nordrh...",50.720188,7.115907,100.00,M,PointAddress
2,20097,"Wendenstraße 8, 20097, Hamburg, Hamburg-Mitte,...",53.547009,10.025767,99.29,M,PointAddress
3,30173,"Freundallee 23, 30173, Hannover, Bult, Nieders...",52.365701,9.773707,100.00,M,PointAddress
4,50668,"Konrad-Adenauer-Ufer 41, 50668, Köln, Altstadt...",50.947820,6.964277,99.35,M,PointAddress
5,85402,"Ringstraße 7, 85402, Kranzberg, Bayern",48.406577,11.609137,100.00,M,PointAddress
6,4155,"Fechnerstraße 8, 04155, Leipzig, Gohlis-Süd, S...",51.358980,12.356217,100.00,M,PointAddress
7,48155,"Martin-Luther-King-Weg 24, 48155, Münster, Mün...",51.934922,7.651978,100.00,M,PointAddress


In [62]:
address_df.merge(results_df, on="id", how="left")

,id,address,postal,city,country,matched_address,latitude,longitude,score,match_status,match_type
0,10178,Karl-Liebknecht-Straße 5,10178,Berlin,DE,"Karl-Liebknecht-Straße 5, 10178, Berlin, Mitte...",52.519724,13.404016,100.00,M,PointAddress
1,53113,Adenauerallee 206,53113,Bonn,DE,"Adenauerallee 206, 53113, Bonn, Gronau, Nordrh...",50.720188,7.115907,100.00,M,PointAddress
2,20097,Wendenstraße 8 - 12,20097,Hamburg,DE,"Wendenstraße 8, 20097, Hamburg, Hamburg-Mitte,...",53.547009,10.025767,99.29,M,PointAddress
3,30173,Freundallee 23,30173,Hannover,DE,"Freundallee 23, 30173, Hannover, Bult, Nieders...",52.365701,9.773707,100.00,M,PointAddress
4,50668,Konrad-Adenauer-Ufer 41-45,50668,Köln,DE,"Konrad-Adenauer-Ufer 41, 50668, Köln, Altstadt...",50.947820,6.964277,99.35,M,PointAddress
5,85402,Ringstraße 7,85402,Kranzberg,DE,"Ringstraße 7, 85402, Kranzberg, Bayern",48.406577,11.609137,100.00,M,PointAddress
6,4155,Fechnerstraße 8,4155,Leipzig,DE,"Fechnerstraße 8, 04155, Leipzig, Gohlis-Süd, S...",51.358980,12.356217,100.00,M,PointAddress
7,48155,Martin-Luther-King-Weg 24,48155,Münster,DE,"Martin-Luther-King-Weg 24, 48155, Münster, Mün...",51.934922,7.651978,100.00,M,PointAddress


## Display Results on a Map

If the geocoding succeeded, you can add the points to a new map view.

In [34]:
# Add points to the map using a spatially-enabled dataframe
results_sdf = pd.DataFrame.spatial.from_xy(results_df, x_column="longitude", y_column="latitude")

# Visualize the spatially-enabled dataframe
results_map_view = portal.map("Bonn, Germany")
results_map_view.basemap.basemap = "osm"
results_sdf.spatial.plot(results_map_view)
results_map_view.zoom_to_layer(results_sdf)
results_map_view

Map()